# Machine Learning Practical: KNN & Naive Bayes

This notebook solves both problems using **Pandas, scikit-learn, and Matplotlib**.

- **Problem 1:** K-Nearest Neighbors (KNN) for diabetes risk classification.
- **Problem 2:** Gaussian Naive Bayes for loan approval prediction.

Run the cells from top to bottom. The notebook prints data-cleaning checks, evaluation metrics, confusion matrices, classification reports, and predictions for the new records.

## Imports

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

# Problem 1: K-Nearest Neighbors (KNN)
## 1. Load the patient diabetes-risk dataset

In [ ]:
patient_data = {
    "Patient_ID": ["P101","P102","P103","P104","P105","P106","P107","P108",
                   "P109","P110","P111","P112","P113","P114","P115"],
    "Age": [25,45,35,52,29,48,40,31,55,38,43,27,50,33,46],
    "BMI": [22.5,31.2,27.5,34.8,24.0,32.5,29.0,23.5,36.0,26.0,30.5,21.8,33.0,25.0,31.8],
    "Glucose": [85,145,110,175,92,160,130,95,185,105,140,88,168,100,155],
    "Blood_Pressure": [70,85,80,95,75,90,85,72,100,78,88,68,92,76,89],
    "Risk": ["Low","High","Low","High","Low","High","High","Low","High","Low","High","Low","High","Low","High"]
}

patients = pd.DataFrame(patient_data)
display(patients)

## 2. Clean data and check missing values / duplicates

In [ ]:
print("Missing values before cleaning:")
print(patients.isnull().sum())

print("\\nDuplicate rows:", patients.duplicated().sum())

# Remove exact duplicate rows, if any
patients = patients.drop_duplicates().copy()

# Fill missing numeric values with their column median and categorical values with mode.
numeric_cols = ["Age", "BMI", "Glucose", "Blood_Pressure"]
for col in numeric_cols:
    patients[col] = patients[col].fillna(patients[col].median())

patients["Risk"] = patients["Risk"].fillna(patients["Risk"].mode()[0])

print("\\nMissing values after cleaning:")
print(patients.isnull().sum())

## 3–6. Separate X and y, encode target, split, and scale

The patient ID is an identifier, so it is not used as a model feature. The target is encoded as **Low = 0** and **High = 1**.

In [ ]:
X = patients[["Age", "BMI", "Glucose", "Blood_Pressure"]]
y = patients["Risk"].map({"Low": 0, "High": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])
print("Training features after scaling:")
display(pd.DataFrame(X_train_scaled, columns=X.columns).head())

## 7–9. Train KNN models for K = 3, 5, and 7

KNN uses distances between records. StandardScaler is important because the features use different measurement scales (for example, age and glucose).

In [ ]:
k_values = [3, 5, 7]
knn_results = []
knn_models = {}

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)

    knn_models[k] = model
    knn_results.append({"K": k, "Accuracy": accuracy})

    print(f"\\nResults for K = {k}")
    print("Accuracy:", round(accuracy, 4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(
        y_test, y_pred, labels=[0, 1],
        target_names=["Low", "High"], zero_division=0
    ))

knn_accuracy_df = pd.DataFrame(knn_results)
print("\\nAccuracy comparison:")
display(knn_accuracy_df)

### Visualize KNN accuracy comparison

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(knn_accuracy_df["K"].astype(str), knn_accuracy_df["Accuracy"])
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy for Different K Values")
plt.ylim(0, 1.05)
plt.show()

# Select the K with the highest test accuracy.
# If there is a tie, choose the smaller K.
best_k = sorted(knn_results, key=lambda row: (-row["Accuracy"], row["K"]))[0]["K"]
best_knn = knn_models[best_k]
print(f"Selected K = {best_k}")

## 10. Confusion matrix and classification report for the selected K

The loop above prints these metrics for every K. This cell displays the selected model's confusion matrix as a plot.

In [ ]:
best_test_predictions = best_knn.predict(X_test_scaled)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_test_predictions,
    labels=[0, 1],
    display_labels=["Low", "High"],
    cmap="Blues"
)
plt.title(f"KNN Confusion Matrix (K={best_k})")
plt.show()

print(classification_report(
    y_test, best_test_predictions,
    labels=[0, 1],
    target_names=["Low", "High"],
    zero_division=0
))

## 11. Predict diabetes risk for new patients

The same scaler fitted on the training data is used to transform new patient measurements. Predictions are shown for all three K values so you can compare them.

In [ ]:
new_patients = pd.DataFrame({
    "Patient_ID": ["N101", "N102", "N103"],
    "Age": [42, 26, 53],
    "BMI": [30.0, 22.0, 35.0],
    "Glucose": [138, 90, 180],
    "Blood_Pressure": [86, 70, 98]
})

new_X_scaled = scaler.transform(
    new_patients[["Age", "BMI", "Glucose", "Blood_Pressure"]]
)

new_patient_results = new_patients[["Patient_ID"]].copy()
for k in k_values:
    predictions = knn_models[k].predict(new_X_scaled)
    new_patient_results[f"Risk_K{k}"] = pd.Series(predictions).map({0: "Low", 1: "High"})

display(new_patient_results)

print(f"Selected model (K={best_k}) predictions:")
new_patient_results[["Patient_ID", f"Risk_K{best_k}"]]

### KNN learning notes
- KNN classifies a new record by looking at the labels of its nearest training records.
- A small K can be sensitive to noise; a larger K can smooth the decision but may miss local patterns.
- Scaling is essential for distance-based methods because large-range features can dominate distance calculations.
- This dataset is very small, so the test accuracy can change with the train/test split. These predictions are for a classroom exercise, not clinical diagnosis.

# Problem 2: Naive Bayes – Banking Loan Approval
## 1. Load the loan approval dataset

In [ ]:
loan_data = {
    "Customer_ID": ["C101","C102","C103","C104","C105","C106","C107","C108",
                    "C109","C110","C111","C112","C113","C114","C115"],
    "Income": ["High","Low","Medium","Low","High","Medium","High","Low",
               "Medium","High","Low","Medium","High","Low","Medium"],
    "Employment": ["Salaried","Self-employed","Salaried","Salaried","Business",
                   "Self-employed","Salaried","Business","Salaried","Business",
                   "Self-employed","Business","Salaried","Salaried","Self-employed"],
    "Credit_Score": [780,580,690,610,750,640,720,590,710,800,620,670,760,550,630],
    "Existing_Loan": ["No","Yes","No","Yes","No","Yes","No","Yes","No","No",
                      "Yes","No","Yes","No","Yes"],
    "Loan_Status": ["Approved","Rejected","Approved","Rejected","Approved","Rejected",
                    "Approved","Rejected","Approved","Approved","Rejected","Approved",
                    "Approved","Rejected","Rejected"]
}

loans = pd.DataFrame(loan_data)
display(loans)

## 2. Check missing values and duplicate records

In [ ]:
print("Missing values:")
print(loans.isnull().sum())

print("\\nDuplicate rows:", loans.duplicated().sum())
loans = loans.drop_duplicates().copy()

# Basic missing-value handling
for col in ["Income", "Employment", "Existing_Loan", "Loan_Status"]:
    loans[col] = loans[col].fillna(loans[col].mode()[0])

loans["Credit_Score"] = loans["Credit_Score"].fillna(loans["Credit_Score"].median())

print("\\nMissing values after cleaning:")
print(loans.isnull().sum())

## 3–5. Separate features and target, encode categories, and split data

We use one-hot encoding for the categorical input columns and keep credit score numeric. A preprocessing pipeline ensures the encoder is fitted only on the training set and applies the same transformations to new customers.

In [ ]:
X_loan = loans[["Income", "Employment", "Credit_Score", "Existing_Loan"]]
y_loan = loans["Loan_Status"]

categorical_features = ["Income", "Employment", "Existing_Loan"]
numeric_features = ["Credit_Score"]

# Dense one-hot output is used because GaussianNB expects dense numeric arrays.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_features),
        ("numeric", "passthrough", numeric_features)
    ],
    remainder="drop"
)

X_train_loan, X_test_loan, y_train_loan, y_test_loan = train_test_split(
    X_loan, y_loan, test_size=0.20, random_state=42, stratify=y_loan
)

print("Training records:", X_train_loan.shape[0])
print("Testing records:", X_test_loan.shape[0])

## 6–8. Build, train, and evaluate Gaussian Naive Bayes

In [ ]:
nb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GaussianNB())
])

nb_model.fit(X_train_loan, y_train_loan)

loan_predictions = nb_model.predict(X_test_loan)
loan_accuracy = accuracy_score(y_test_loan, loan_predictions)

print("Gaussian Naive Bayes Accuracy:", round(loan_accuracy, 4))

## 9–10. Confusion matrix and classification report

In [ ]:
loan_labels = ["Approved", "Rejected"]

print("Confusion Matrix:")
print(confusion_matrix(y_test_loan, loan_predictions, labels=loan_labels))

print("\\nClassification Report:")
print(classification_report(
    y_test_loan, loan_predictions,
    labels=loan_labels,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test_loan,
    loan_predictions,
    labels=loan_labels,
    cmap="Blues"
)
plt.title("Gaussian Naive Bayes – Loan Approval")
plt.show()

## 11. Predict loan status for new customers

Customer N204 has a missing Existing Loan value in the question. To make a prediction, this notebook assumes **Existing_Loan = "No"** for N204. Change that value if the correct information is available.

In [ ]:
new_customers = pd.DataFrame({
    "Customer_ID": ["N201", "N202", "N203", "N204"],
    "Income": ["High", "Low", "Medium", "High"],
    "Employment": ["Salaried", "Self-employed", "Business", "Business"],
    "Credit_Score": [740, 600, 680, 650],
    "Existing_Loan": ["No", "Yes", "No", "No"]  # N204 assumed "No"
})

new_customer_predictions = nb_model.predict(
    new_customers[["Income", "Employment", "Credit_Score", "Existing_Loan"]]
)

loan_result = new_customers[["Customer_ID"]].copy()
loan_result["Predicted_Loan_Status"] = new_customer_predictions
display(loan_result)

## Final learning summary

### KNN
- Uses distance to classify observations.
- Feature scaling is important.
- Compare multiple K values rather than assuming one value is always best.

### Gaussian Naive Bayes
- Uses Bayes' theorem and assumes features are conditionally independent given the class.
- GaussianNB models continuous numeric features such as credit score with a Gaussian distribution.
- Categorical features are converted to numeric indicator columns using one-hot encoding.

**Important:** Both datasets contain only 15 records. Accuracy and predictions are therefore illustrative and should not be treated as reliable real-world healthcare or banking decisions.